# Stage-1 HPO Winner Analysis

Quick analysis notebook for Stage-1 HPO runs (QBC or Marker).
It loads `matrix.tsv` + per-run `qbc/history.jsonl`, summarizes metrics, and ranks candidate configs.


In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 140)
pd.set_option('display.max_columns', 200)


In [ ]:
# --- Configure ---
REPO_ROOT = Path.cwd()
METHOD = 'qbc_deep_ensemble'  # 'qbc_deep_ensemble' | 'marker_directed'
STAGE = 'stage1_policy_search'
HPO_ID = None  # set explicit folder name if needed, else latest folder is used

HPO_BASE = REPO_ROOT / 'outputs' / 'hpo' / METHOD / STAGE
if not HPO_BASE.exists():
    raise FileNotFoundError(f'HPO path not found: {HPO_BASE}')

if HPO_ID is None:
    candidates = sorted([p for p in HPO_BASE.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise RuntimeError(f'No HPO runs found in: {HPO_BASE}')
    HPO_RUN = candidates[-1]
else:
    HPO_RUN = HPO_BASE / HPO_ID

MATRIX_PATH = HPO_RUN / 'matrix.tsv'
if not MATRIX_PATH.exists():
    raise FileNotFoundError(f'Matrix not found: {MATRIX_PATH}')

print('HPO_RUN:', HPO_RUN)
print('MATRIX_PATH:', MATRIX_PATH)


In [ ]:
def _parse_override_blob(blob: str) -> dict[str, str]:
    out = {}
    text = (blob or '').strip()
    if not text:
        return out
    for part in text.split(';;'):
        if '=' not in part:
            continue
        k, v = part.split('=', 1)
        out[k] = v
    return out

def _load_last_history_row(history_path: Path) -> dict:
    if not history_path.exists():
        return {}
    last = None
    with history_path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            last = json.loads(line)
    return last or {}

def _as_float_or_nan(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan


In [ ]:
matrix = pd.read_csv(MATRIX_PATH, sep='\t')
rows = []

for _, r in matrix.iterrows():
    run_root = Path(r['run_root'])
    status_path = run_root / 'hpo_status.json'
    manifest_path = run_root / 'run_manifest.json'
    history_path = run_root / 'qbc' / 'history.jsonl'

    status = {}
    if status_path.exists():
        status = json.loads(status_path.read_text(encoding='utf-8'))

    manifest = {}
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))

    last_hist = _load_last_history_row(history_path)

    stage1_overrides = _parse_override_blob(r.get('stage1_overrides', ''))

    rows.append({
        'cfg_id': r['cfg_id'],
        'row_idx': int(r['row_idx']),
        'seed': r['seed'],
        'budget': r['budget'],
        'run_root': str(run_root),
        'status_code': status.get('return_code', np.nan),
        'status_ok': (status.get('return_code', 1) == 0),
        'final_round_idx': last_hist.get('round_idx', np.nan),
        'final_train_size': last_hist.get('train_size', np.nan),
        'final_eval_rmse': _as_float_or_nan(last_hist.get('eval_rmse')),
        'final_eval_mse': _as_float_or_nan(last_hist.get('eval_mse')),
        'final_selected_mean_score': _as_float_or_nan(last_hist.get('selected_mean_score')),
        'final_mean_score': _as_float_or_nan(last_hist.get('mean_score')),
        'stage1_overrides': r.get('stage1_overrides', ''),
        'config_signature': r.get('stage1_overrides', ''),
        **{f'ov::{k}': v for k, v in stage1_overrides.items()},
    })

df = pd.DataFrame(rows)
print('rows:', len(df))
print('successful rows:', int(df['status_ok'].sum()))
df.head(3)


In [ ]:
# Failures / incomplete rows
display(df.loc[~df['status_ok'], ['cfg_id','seed','budget','status_code','run_root']].head(20))


In [ ]:
# Decide ranking metric
metric_candidates = ['final_eval_rmse', 'final_eval_mse', 'final_selected_mean_score']
available = [m for m in metric_candidates if df[m].notna().any()]
print('Available ranking metrics:', available)

if 'final_eval_rmse' in available:
    RANK_METRIC = 'final_eval_rmse'
    ASCENDING = True
elif 'final_eval_mse' in available:
    RANK_METRIC = 'final_eval_mse'
    ASCENDING = True
else:
    RANK_METRIC = 'final_selected_mean_score'
    ASCENDING = False

print('Using ranking metric:', RANK_METRIC, '| ascending:', ASCENDING)


In [ ]:
# Aggregate by configuration (across seeds)
agg = (
    df[df['status_ok']]
    .groupby('config_signature', as_index=False)
    .agg(
        n_runs=('cfg_id', 'count'),
        n_seeds=('seed', 'nunique'),
        mean_metric=(RANK_METRIC, 'mean'),
        std_metric=(RANK_METRIC, 'std'),
        min_metric=(RANK_METRIC, 'min'),
        max_metric=(RANK_METRIC, 'max'),
    )
    .sort_values(['mean_metric','std_metric'], ascending=[ASCENDING, True])
    .reset_index(drop=True)
)

top = agg.head(10)
display(top)

WINNER_SIGNATURE = top.iloc[0]['config_signature'] if len(top) else None
print('Winner signature:', WINNER_SIGNATURE)


In [ ]:
# Seed-level consistency for top configs
top_signatures = agg.head(5)['config_signature'].tolist()
seed_view = (
    df[df['config_signature'].isin(top_signatures)][['cfg_id','seed','budget','config_signature',RANK_METRIC,'status_ok']]
    .sort_values(['config_signature','seed'])
)
display(seed_view)


In [ ]:
# Quick plot for top-5 configs
plot_df = df[df['config_signature'].isin(top_signatures) & df[RANK_METRIC].notna()].copy()
if len(plot_df):
    fig, ax = plt.subplots(figsize=(12, 4))
    order = list(agg.head(5)['config_signature'])
    data = [plot_df.loc[plot_df['config_signature'] == s, RANK_METRIC].values for s in order]
    ax.boxplot(data, labels=[f'cfg{i+1}' for i in range(len(order))], showmeans=True)
    ax.set_title(f'Top-5 config distribution by seed ({RANK_METRIC})')
    ax.set_ylabel(RANK_METRIC)
    ax.grid(True, axis='y', alpha=0.3)
    plt.show()
else:
    print('No numeric metric values available for plotting.')


In [ ]:
# Export winner tables
summary_dir = HPO_RUN / 'summary'
summary_dir.mkdir(parents=True, exist_ok=True)

all_rows_path = summary_dir / 'stage1_rows_summary.csv'
agg_path = summary_dir / 'stage1_config_ranking.csv'
winner_path = summary_dir / 'stage1_winner.csv'

df.to_csv(all_rows_path, index=False)
agg.to_csv(agg_path, index=False)
if len(agg):
    agg.head(1).to_csv(winner_path, index=False)

print('Wrote:', all_rows_path)
print('Wrote:', agg_path)
if len(agg):
    print('Wrote:', winner_path)
